# Streaming Model Output

- In LLMs streamig means the model starts sending tokens/outputs as soon as they're generated, instead of waiting for the entire response to be ready before returing it.
- Why Streaming :
    - Faster response time
    - Mimic human like conversation
    - Important for multimodal UIs
    - Better Ux for long output
    - You can cancel midway, saving tokens
    - You can interleavce UI Updates showing thinking, using tools.

In [1]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gemma3:4b",
    model_provider="ollama",
    temperature = 0
    )

# Without streaming

In [2]:
user_input = 'write a 200 words essay on Impact of AI In Environment'

response = llm.invoke(user_input)
print(response.content)

Okay, here's a 200-word essay on the impact of AI in the environment, aiming for a balanced perspective:

---

Artificial intelligence is rapidly emerging as a powerful tool with the potential to dramatically reshape our approach to environmental challenges. While concerns about its energy consumption are valid, AI’s impact is increasingly leaning towards positive change. 

One of the most significant areas is environmental monitoring. AI-powered systems analyze satellite imagery, sensor data, and drone footage to track deforestation, monitor wildlife populations, and assess pollution levels with unprecedented accuracy and speed. This allows for quicker responses to environmental threats. 

Furthermore, AI is optimizing resource management. Smart grids powered by AI reduce energy waste, while precision agriculture utilizes AI to tailor irrigation and fertilization, minimizing water and chemical usage.  Companies are employing AI to design more sustainable materials and processes, and e

# With Streaming

In [3]:
for chunk in llm.stream(user_input):
    if chunk.content:
        print(chunk.content, end='')


Okay, here's a 200-word essay on the impact of AI in the environment, aiming for a balanced perspective:

---

Artificial intelligence is rapidly emerging as a powerful tool with the potential to dramatically reshape our approach to environmental challenges. While concerns about its energy consumption are valid, AI’s impact is increasingly leaning towards positive change. 

One of the most significant areas is environmental monitoring. AI-powered systems can analyze satellite imagery and sensor data to detect deforestation, track wildlife populations, and predict natural disasters with unprecedented accuracy, allowing for quicker and more targeted responses. Furthermore, AI is optimizing resource management – from smart irrigation systems reducing water waste to algorithms predicting energy demand and improving grid efficiency. 

However, the development and deployment of AI itself isn’t without its drawbacks. Training complex AI models requires substantial energy, and the production o

# Structured Output Streaming

In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

In [5]:
from pydantic import BaseModel, Field
from typing import Annotated,Optional,Literal,TypedDict,List,Dict

In [ ]:
planner_model = init_chat_model(
    model="glm-5:cloud",
    model_provider="ollama",
    temperature = 0
    )

In [42]:
planner_model = init_chat_model(
    model="gemma3:4b",
    model_provider="ollama",
    temperature = 0
    )

planner_model.invoke('hi')


AIMessage(content="Hi there! How's your day going so far? Is there anything you'd like to chat about, or were you just saying hello? 😊 \n\nI'm here to help with just about anything – answering questions, brainstorming ideas, writing stories, or just having a conversation.", additional_kwargs={}, response_metadata={'model': 'gemma3:4b', 'created_at': '2026-03-06T19:09:10.017124Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6089246500, 'load_duration': 3121737300, 'prompt_eval_count': 10, 'prompt_eval_duration': 144477500, 'eval_count': 60, 'eval_duration': 2583763800, 'logprobs': None, 'model_name': 'gemma3:4b', 'model_provider': 'ollama'}, id='lc_run--019cc48d-a836-77a3-bb77-b804ea743f88-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 60, 'total_tokens': 70})

In [54]:
class PlannerStepSchema(BaseModel):
    step_no : int = Field(description='Denotes the current step number')
    robot_type : Literal["ARM","AGV"] = Field(description='Robot needed for executing the action')
    action : str = Field(description='The action to perform')
    dependencies : Optional[List[int]] = Field(description='Any dependency with previous step')
    

class PlannerSchema(BaseModel):
    plan : List[PlannerStepSchema]

In [55]:
def get_planner_prompt():
    prompt = """
    System Role: You are a Robotic Mission Planner. Your task is to decompose a complex mission into a sequence of "action"able primitives for a heterogeneous fleet of robots.
    
    The Mission: {task}

    The Robot Fleet: {robots}

    Environment Context: {locations_info}
    
    When Generating Plan :
        - Make sure that you plan does not conflict with any information you have.
        - Always keep in mind robots locations and capabilities.
        - Multi-Parent Logic: If an "action" involves two entities (e.g., an Arm placing an item on an "AGV", the step MUST depend on BOTH the Arm's previous step AND the"AGV"s previous step.
        - Gripper State: A 'place' "action" MUST depend on the 'pick' "action" for that specific object.
        - Location State: An "action" at a location MUST depend on the 'navigate' "action" to that location.
        - Understand the dependency concept from the below example.
    
    Example:
        task = 'move ball and pen to drop zone 1'
        
        {{
            "plan" : [
                        [
                            "step_no" : 1,
                            "robot_type" : "AGV",
                            "action" : "agv_1 navigates to Delivery Zone",
                            "dependencies" : []
                        ],
                        [
                            "step_no" : 2,
                            "robot_type" : "ARM",
                            "action" : "arm_1 picks ball",
                            "dependencies" : []
                        ],
                        [
                            "step_no" : 3,
                            "robot_type" : "ARM",
                            "action" : "arm_1 place ball on agv_1",
                            "dependencies" : [1,2]
                        ],
                        [
                            "step_no" : 4,
                            "robot_type" : "AGV",
                            "action" : "agv_1 navigates to Drop Zone 1",
                            "dependencies" : [3]
                        ],
                        [
                            "step_no" : 5,
                            "robot_type" : "AGV",
                            "action" : "agv_2 navigates to Delivery Zone",
                            "dependencies" : []
                        ],
                        [
                            "step_no" : 6,
                            "robot_type" : "ARM",
                            "action" : "arm_1 picks pen"
                            "dependencies" : [3]
                        ],
                        [
                            "step_no" : 7,
                            "robot_type" : "ARM",
                            "action" : "arm_1 place pen on agv_2",
                            "dependencies" : [5,6]
                        ],
                        [
                            "step_no" : 8,
                            "robot_type" : "AGV",
                            "action" : "agv_2 navigates to Drop Zone 1",
                            "dependencies" : [7]
                        ]
                    ]
        }}
        
        IMPORTANT: Return ONLY valid JSON. Do not include any variable assignments (like 'plan =') or single quotes.
    """
    return prompt

In [56]:
ROBOTS = [
    {
        'name' : ['arm_1'],
        'type' : 'ARM',
        'description' : 'ARM is capable of picking and placing objects but is static at Delivery Zone and cannot navigate to other locations'
    },
    {
        'name' : ['agv_1','agv_2'],
        'type' : 'AGV',
        'description' : 'AGV is capable of navigating to Delivery Zone, Drop Zone 1, Drop Zone 2, but is not capable of picking or placing objects'
    }
    ]

OBJECTS = ['ball', 'pen', 'glass', 'mouse']
LOCATIONS = ['Delivery Zone','Drop Zone 1', 'Drop Zone 2']

LOCATION_INFO = [
    {
        'name' : 'Delivery Zone',
        'description' : 'All objects from object list are located here for pickup and only the arm_1 is located here'
    },
    {
        'name' : 'Drop Zone 1',
        'description' : 'Location for agv_1 or agv_2 to drop the objects, separate from other locations.'
    },
    {
        'name' : 'Drop Zone 2',
        'description' : 'Location for agv_1 or agv_2 to drop the objects, separate from other locations.'
    },
    ]


In [57]:
def plannerLLM():
    
    task = 'move ball to drop zone 1'
    model = planner_model.with_structured_output(PlannerSchema)
    
    template = PromptTemplate(
        template=get_planner_prompt(),
        input_variables=['task','robots','objects','locations','locations_info']
    )
    
    chain = template | model
    
    response = chain.invoke(
        {
            'task' : task,
            'robots' : ROBOTS,
            'locations_info' : LOCATION_INFO,
        }
    )
    print('________________________ From Planner : ')
    print('Plan Generated :')
    print()
    for step in response.plan:
        print(step)

plannerLLM()

________________________ From Planner : 
Plan Generated :

step_no=1 robot_type='AGV' action='agv_1 navigates to Delivery Zone' dependencies=[]
step_no=2 robot_type='ARM' action='arm_1 picks ball' dependencies=[]
step_no=3 robot_type='ARM' action='arm_1 place ball on agv_1' dependencies=[1, 2]
step_no=4 robot_type='AGV' action='agv_1 navigates to Drop Zone 1' dependencies=[3]


In [60]:
from langchain_core.output_parsers import PydanticOutputParser

def plannerLLM():
    
    task = 'move ball to drop zone 1'
    
    model = planner_model
    
    parser = PydanticOutputParser(pydantic_object=PlannerSchema)
    
    template = PromptTemplate(
        template=get_planner_prompt(),
        input_variables=['task','robots','objects','locations','locations_info']
    )
    
    
    prompt = template.invoke(
        {
            'task' : task,
            'robots' : ROBOTS,
            'locations_info' : LOCATION_INFO,
        }
    )
    
    full_content = ''
    for chunk in model.stream(prompt):
        if chunk.content:
            full_content += chunk.content
            print(chunk.content, end = '')
    
    print('---------------------------------------------------------------------')
    print()
    plan_parsed = parser.parse(full_content)
    
    plan = plan_parsed.plan
    for step in plan:
        print(step)
    print(plan_parsed)

plannerLLM()

```json
{
    "plan" : [
        {
            "step_no" : 1,
            "robot_type" : "AGV",
            "action" : "agv_1 navigates to Delivery Zone",
            "dependencies" : []
        },
        {
            "step_no" : 2,
            "robot_type" : "ARM",
            "action" : "arm_1 picks ball",
            "dependencies" : []
        },
        {
            "step_no" : 3,
            "robot_type" : "ARM",
            "action" : "arm_1 place ball on agv_1",
            "dependencies" : [1, 2]
        },
        {
            "step_no" : 4,
            "robot_type" : "AGV",
            "action" : "agv_1 navigates to Drop Zone 1",
            "dependencies" : [3]
        }
    ]
}
```---------------------------------------------------------------------

step_no=1 robot_type='AGV' action='agv_1 navigates to Delivery Zone' dependencies=[]
step_no=2 robot_type='ARM' action='arm_1 picks ball' dependencies=[]
step_no=3 robot_type='ARM' action='arm_1 place ball on agv_1' depend